# Week 6 · Day 5 — Apple Inc. Stock Statistical Analysis

**Mini-project:** NumPy, Pandas, SciPy and Matplotlib  
**Dataset:** Apple Stock Prices From 1981 to 2023 (course dataset)

**Research questions:** How did closing prices and trading activity change over time? What does the distribution of daily returns look like? How do annual closing-price means differ? How are price, volume, and moving-average indicators related?

> Historical observational analysis only. A significant statistical test does not establish a trading advantage or predict future prices.

## 1. Data Loading and Exploration

Load the course ZIP, inspect columns, missing values, duplicate dates, and trading-day spacing. The loader accepts common CSV naming conventions and adjusted-close aliases.

In [ ]:
from pathlib import Path
from io import BytesIO
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import stats, signal
from IPython.display import display, Markdown

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
ROOT = Path.cwd()
OUT = ROOT / 'results'
OUT.mkdir(exist_ok=True)
DATA_URL = ('https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/'
            'Week%203/W3D4%20-%20Mini%20Project/Apple%20Stock%20Prices%20From%201981%20to%202023.zip')
print('Working directory:', ROOT)
print('Output directory:', OUT)


In [ ]:
def read_apple_dataset():
    # Prefer a locally saved CSV/ZIP, then download the course archive.
    local_csvs = [f for f in ROOT.glob('*.csv') if 'apple' in f.name.lower() or 'aapl' in f.name.lower()]
    if local_csvs:
        print('Loading local CSV:', local_csvs[0].name)
        return pd.read_csv(local_csvs[0])
    local_zips = list(ROOT.glob('*.zip'))
    if local_zips:
        payload = local_zips[0].read_bytes()
        print('Loading local ZIP:', local_zips[0].name)
    else:
        print('Downloading Apple stock dataset...')
        try:
            with urllib.request.urlopen(DATA_URL, timeout=45) as response:
                payload = response.read()
        except Exception as exc:
            raise RuntimeError('Download failed. Save the course ZIP beside the notebook and run again.') from exc
    try:
        with zipfile.ZipFile(BytesIO(payload)) as archive:
            csvs = [x for x in archive.namelist() if x.lower().endswith('.csv') and not x.startswith('__MACOSX')]
            if not csvs:
                raise ValueError('No CSV found inside ZIP. Extract the data as CSV beside this notebook.')
            print('Reading:', csvs[0])
            with archive.open(csvs[0]) as file:
                return pd.read_csv(file)
    except zipfile.BadZipFile as exc:
        raise RuntimeError('Downloaded file was not a valid ZIP. Download the dataset manually.') from exc

raw = read_apple_dataset()
print('Raw shape:', raw.shape)
print('Raw columns:', raw.columns.tolist())
print('Missing values:')
print(raw.isna().sum().to_string())
print('Data types:')
print(raw.dtypes.to_string())
display(raw.head())


### Cleaning decisions

- Normalize column names, coerce numeric fields, and parse dates. Invalid dates and rows missing essential OHLC/volume values are excluded from analyses requiring those values.
- Remove duplicate trading dates (keep the first record) and sort chronologically.
- Do **not** fill absent stock prices with zero: this would introduce artificial market movements.
- Keep adjusted closing prices separately when supplied. Corporate actions such as splits can distort comparisons of unadjusted prices over long periods.

In [ ]:
def canonical(col):
    return ''.join(ch for ch in str(col).lower() if ch.isalnum())

aliases = {'date': 'Date', 'open': 'Open', 'high': 'High', 'low': 'Low',
           'close': 'Close', 'adjclose': 'Adj Close', 'adjustedclose': 'Adj Close',
           'volume': 'Volume'}
rename = {col: aliases[canonical(col)] for col in raw.columns if canonical(col) in aliases}
df = raw.rename(columns=rename).copy()
required = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
missing_cols = sorted(set(required) - set(df.columns))
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}; available: {df.columns.tolist()}')
initial_rows = len(df)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
for c in ['Open', 'High', 'Low', 'Close', 'Volume', 'Adj Close']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', '', regex=False), errors='coerce')
df = df.dropna(subset=required).drop_duplicates(subset='Date').sort_values('Date').reset_index(drop=True)
df = df[(df[['Open', 'High', 'Low', 'Close']] > 0).all(axis=1) & (df['Volume'] >= 0)].copy()
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.to_period('M')
df['Daily Return'] = df['Close'].pct_change(fill_method=None)
if 'Adj Close' in df and df['Adj Close'].notna().sum() > 10:
    df['Adjusted Return'] = df['Adj Close'].pct_change(fill_method=None)
    return_col = 'Adjusted Return'
else:
    return_col = 'Daily Return'
print('Rows removed:', initial_rows - len(df))
print('Clean shape:', df.shape)
print('Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())
print('Day gaps (most common):')
print(df['Date'].diff().dt.days.value_counts().head().to_string())
print('Return measure:', return_col)
display(df.head())


## 2. Data Visualization

Time-series prices and volume are plotted separately because they have different units. The candlestick chart displays a representative recent window, with vertical lines for high–low and rectangular bodies for open–close.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(df['Date'], df['Close'], color='navy', linewidth=1)
axes[0].set(title='Apple Daily Closing Price', ylabel='Price (USD, as recorded)')
axes[1].plot(df['Date'], df['Volume'], color='darkorange', linewidth=.7)
axes[1].set(title='Daily Trading Volume', ylabel='Shares', xlabel='Date')
plt.tight_layout()
plt.savefig(OUT / '01_price_and_volume.png', dpi=150)
plt.show()


In [ ]:
window = df.tail(min(60, len(df))).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(15, 6))
for i, r in window.iterrows():
    color = '#18835d' if r['Close'] >= r['Open'] else '#bd4646'
    ax.vlines(i, r['Low'], r['High'], color=color, linewidth=1)
    bottom = min(r['Open'], r['Close'])
    height = max(abs(r['Close'] - r['Open']), .00001)
    ax.add_patch(Rectangle((i - .3, bottom), .6, height, facecolor=color, edgecolor=color))
ax.set_xlim(-1, len(window))
ticks = np.arange(0, len(window), max(1, len(window)//10))
ax.set_xticks(ticks, window.loc[ticks, 'Date'].dt.strftime('%Y-%m-%d'), rotation=45, ha='right')
ax.set(title='Apple Candlestick Chart — Last 60 Trading Days', xlabel='Date', ylabel='Price (USD)')
plt.tight_layout()
plt.savefig(OUT / '02_candlestick.png', dpi=150)
plt.show()


## 3. Statistical Analysis

Compare central tendency and dispersion. Moving averages smooth short-term price fluctuations but lag the underlying series.

In [ ]:
summary = df[['Open', 'High', 'Low', 'Close', 'Volume']].agg(['count', 'mean', 'median', 'std', 'min', 'max']).T
print('Summary statistics:')
display(summary)
summary.to_csv(OUT / 'summary_statistics.csv')
for days in [20, 50, 200]:
    df[f'MA{days}'] = df['Close'].rolling(days, min_periods=days).mean()
fig, ax = plt.subplots(figsize=(13, 5))
recent = df.tail(min(750, len(df)))
ax.plot(recent['Date'], recent['Close'], label='Close', alpha=.55)
for days in [20, 50, 200]:
    ax.plot(recent['Date'], recent[f'MA{days}'], label=f'{days}-day MA')
ax.set(title='Apple Closing Price and Moving Averages — Recent Trading Days', xlabel='Date', ylabel='USD')
ax.legend()
plt.tight_layout()
plt.savefig(OUT / '03_moving_averages.png', dpi=150)
plt.show()


## 4. Hypothesis Testing

### 4a. Comparing closing prices between years

**Null hypothesis (H₀):** Mean daily closing prices are equal for the two most recent complete years in the dataset.  
**Alternative (H₁):** The means differ.  
Use Welch's independent-samples t-test, which does not assume equal variances. **Caution:** Daily stock prices are serially correlated and often nonstationary, so this test's independence assumption is weak; interpret the p-value as a descriptive classroom exercise, not proof of a structural change.


In [ ]:
counts = df.groupby('Year').size()
full_years = counts[counts >= 180].index.sort_values()
if len(full_years) < 2:
    raise ValueError('At least two years with 180+ trading days are required.')
y1, y2 = int(full_years[-2]), int(full_years[-1])
a = df.loc[df['Year'] == y1, 'Close'].to_numpy()
b = df.loc[df['Year'] == y2, 'Close'].to_numpy()
t_result = stats.ttest_ind(a, b, equal_var=False)
print(f'Years: {y1} (n={len(a)}) vs {y2} (n={len(b)})')
print(f'Mean closes: {a.mean():.3f} vs {b.mean():.3f} USD')
print(f'Welch t = {t_result.statistic:.4f}, p = {t_result.pvalue:.6g}')
print('At alpha=0.05:', 'reject H0' if t_result.pvalue < .05 else 'do not reject H0')


### 4b. Daily return distribution and normality

Test whether daily returns are consistent with a normal distribution using SciPy's D’Agostino–Pearson normality test. A small p-value indicates evidence against normality. Financial return distributions can have heavy tails and volatility clustering; a normality test alone does not capture these time-dependent features.

In [ ]:
returns = df[return_col].replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
normality = stats.normaltest(returns)
print('Returns count:', len(returns))
print('Mean:', np.mean(returns), 'Std:', np.std(returns, ddof=1))
print('Skewness:', stats.skew(returns), 'Excess kurtosis:', stats.kurtosis(returns))
print(f'Normality statistic = {normality.statistic:.4f}, p = {normality.pvalue:.6g}')
print('At alpha=0.05:', 'reject normality' if normality.pvalue < .05 else 'do not reject normality')
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(returns * 100, bins=75, density=True, alpha=.7, label='Observed')
xs = np.linspace(np.percentile(returns*100, .5), np.percentile(returns*100, 99.5), 300)
axes[0].plot(xs, stats.norm.pdf(xs, loc=np.mean(returns)*100, scale=np.std(returns, ddof=1)*100), label='Normal reference')
axes[0].set(title='Daily Return Distribution', xlabel='Daily return (%)', ylabel='Density')
axes[0].legend()
stats.probplot(returns, dist='norm', plot=axes[1])
axes[1].set_title('Normal Q–Q Plot')
plt.tight_layout()
plt.savefig(OUT / '04_return_distribution.png', dpi=150)
plt.show()


## 5. Advanced Statistical Techniques (Bonus)

Use NumPy convolution for a moving average and SciPy signal filtering for an additional smoothed price series. These are descriptive smoothing tools, not forecasts. Compute correlation between price, volume, and moving-average features. Correlations between trending levels can be spurious; daily percentage changes are often more meaningful.

In [ ]:
close = df['Close'].to_numpy(dtype=float)
window_size = 20
convolved = np.convolve(close, np.ones(window_size)/window_size, mode='valid')
rolling = df['MA20'].dropna().to_numpy()
print('NumPy convolution matches Pandas rolling mean:', np.allclose(convolved, rolling))
# Savitzky–Golay smoothing requires an odd window length and enough observations.
sg_window = min(51, len(close) if len(close)%2 else len(close)-1)
if sg_window >= 5:
    df['SavGol'] = signal.savgol_filter(close, window_length=sg_window, polyorder=2)
    fig, ax = plt.subplots(figsize=(13, 5))
    view = df.tail(min(500, len(df)))
    ax.plot(view['Date'], view['Close'], label='Close', alpha=.5)
    ax.plot(view['Date'], view['SavGol'], label='Savitzky–Golay smoothed')
    ax.set(title='SciPy Signal Processing — Smoothed Closing Prices', ylabel='USD')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT / '05_signal_processing.png', dpi=150)
    plt.show()


In [ ]:
corr_frame = df[['Close', 'Volume', 'MA20', 'MA50', 'MA200']].dropna()
corr = np.corrcoef(corr_frame.to_numpy().T)
corr_df = pd.DataFrame(corr, index=corr_frame.columns, columns=corr_frame.columns)
print('Correlation of price/volume/moving-average levels:')
display(corr_df.round(3))
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(corr_df.columns)), corr_df.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr_df.columns)), corr_df.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, ax=ax, label='Pearson correlation')
ax.set_title('Correlation Between Stock Metrics')
plt.tight_layout()
plt.savefig(OUT / '06_correlations.png', dpi=150)
plt.show()
# Compare price and volume daily changes, avoiding trend-driven level correlations.
changes = df[['Close', 'Volume']].pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan).dropna()
print('Daily percentage-change correlation (close vs volume):', changes.corr().iloc[0,1])


## 6. Summary and Insights

The following cell builds a findings summary from the dataset actually loaded, avoiding prewritten or fabricated statistics.

In [ ]:
yearly = df.groupby('Year').agg(mean_close=('Close', 'mean'), total_volume=('Volume', 'sum'), trading_days=('Close', 'size'))
yearly.to_csv(OUT / 'yearly_summary.csv')
summary_text = f"""# Apple Stock Analysis — Findings

- **Coverage:** {df['Date'].min():%Y-%m-%d} through {df['Date'].max():%Y-%m-%d}; {len(df):,} trading-day records.
- **Average closing price:** ${df['Close'].mean():,.2f}; median ${df['Close'].median():,.2f}; standard deviation ${df['Close'].std():,.2f}.
- **Year comparison:** {y1} mean ${a.mean():,.2f} vs {y2} mean ${b.mean():,.2f}; Welch p-value {t_result.pvalue:.4g}. {'The test rejects equal means at 5% significance.' if t_result.pvalue < .05 else 'The test does not reject equal means at 5% significance.'} Independence assumptions are imperfect for price series.
- **Daily returns:** mean {np.mean(returns)*100:.3f}%, standard deviation {np.std(returns,ddof=1)*100:.3f}%; normality-test p-value {normality.pvalue:.4g}. {'Returns depart significantly from normality.' if normality.pvalue < .05 else 'No statistically significant departure from normality was detected.'}
- **Price-volume change correlation:** {changes.corr().iloc[0,1]:.3f} (Pearson); association is not causation.

**Limitations:** Historical observational data, possible corporate-action adjustments, trading-day dependence, nonstationary prices, and no predictive or causal claims. This is not investment advice.
"""
display(Markdown(summary_text))
(OUT / 'analysis_report.md').write_text(summary_text, encoding='utf-8')
print('Saved report and charts to:', OUT)


## 7. Reflection

**Challenges and solutions:**

1. **Data quality and date handling:** Normalize column names, convert dates, sort trading days, and exclude incomplete records without inventing prices.
2. **Comparing long-run prices:** Recognize that stock splits and other corporate actions can distort raw price-level comparisons; use adjusted-close returns where available.
3. **Hypothesis assumptions:** A simple t-test of daily closing prices is easy to compute but violates ideal independence assumptions; report that limitation explicitly.
4. **Normality and outliers:** Compare a return histogram, normal-reference curve, Q–Q plot, and formal test rather than relying on a single summary statistic.
5. **Smoothing and correlations:** Moving averages reduce noise but lag the series, while level correlations may reflect shared time trends rather than meaningful relationships.

**What I learned:** NumPy provides efficient array statistics and convolution; Pandas supports time-series cleaning and grouping; SciPy supplies formal statistical tests and signal processing; Matplotlib makes financial patterns visible. Further work could compare adjusted returns across market regimes and use methods that account for autocorrelation and changing volatility.